# Num. Epoch = 5

## Tests with P(X,Y) instead only P(X) in the distribution distance

## Px - scaling 0

In [5]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------



import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from your filenames
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Px_scaling0/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# sanity check
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Determine number of classes U once, from all labels
# ----------------------------------------------------------------------
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# ----------------------------------------------------------------------
# Build joint Gaussian (mean, variance) summaries for P(X,Y)
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)        # (n, d_x)
    y = labels[k].astype(int)                          # (n,)
    # one-hot encode Y
    Y = np.eye(U, dtype=np.float64)[y]                 # (n, U)
    # joint samples Z = [X; Y]
    Z = np.concatenate([X, Y], axis=1)                 # (n, d_x + U)
    mu  = Z.mean(axis=0)                               # (d_x+U,)
    var = Z.var(axis=0)                                # (d_x+U,)
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """W2 between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    dm2 = np.sum((mu1 - mu2)**2)
    ds2 = np.sum((np.sqrt(var1) - np.sqrt(var2))**2)
    return np.sqrt(dm2 + ds2)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    # Euclidean on your saved descriptor profiles
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    # reference W2 on P(X,Y)
    mu1, var1 = gaussians[k1]
    mu2, var2 = gaussians[k2]
    ref_dist   = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print(f"\nWasserstein-2( P(X,Y) ) ε over {len(epsilons)} pairs")
print("-" * 40)
print(f"max ε   : {epsilons.max():.6f}")
print(f"mean ε  : {epsilons.mean():.6f}")
print(f"median ε: {np.median(epsilons):.6f}")
print(f"std ε   : {epsilons.std():.6f}")
print(f"min ε   : {epsilons.min():.6f}")

pairs: 100%|██████████| 44850/44850 [00:01<00:00, 24245.83it/s]


Wasserstein-2( P(X,Y) ) ε over 44850 pairs
----------------------------------------
max ε   : 1.838569
mean ε  : 0.565347
median ε: 0.564172
std ε   : 0.361669
min ε   : 0.000048


In [6]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Px_scaling0/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")


def flatten(x):
    # returns shape (n_samples, n_features)
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# Determine number of classes
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# Build joint histograms
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50
bins  = np.linspace(vmin, vmax, nbins+1)

histograms = {}
for k in descriptors:
    X_flat = flatten(features[k]).astype(np.float64)  # (n_samples, d)
    y      = labels[k].astype(int)                    # (n_samples,)
    joint_counts = np.zeros((U, nbins), dtype=np.float64)

    for u in range(U):
        mask = (y == u)           # shape (n_samples,)
        if not mask.any():
            continue
        # only flatten the samples of class u
        data_u = X_flat[mask].ravel()  
        counts, _ = np.histogram(data_u, bins=bins)
        joint_counts[u] = counts

    prob = joint_counts.ravel()
    total = prob.sum()
    if total > 0:
        prob /= total
    histograms[k] = prob

# Compute ε_js over all pairs
from scipy.spatial.distance import jensenshannon
import itertools
from tqdm import tqdm

pairs  = list(itertools.combinations(descriptors.keys(), 2))
eps_js = []

for k1, k2 in tqdm(pairs, desc="JSD P(X,Y) pairs"):
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])
    p  = histograms[k1]
    q  = histograms[k2]
    js = jensenshannon(p, q)
    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print(f"\nJensen–Shannon ε over P(X,Y) for {len(eps_js)} pairs")
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD P(X,Y) pairs: 100%|██████████| 44850/44850 [00:02<00:00, 17695.80it/s]


Jensen–Shannon ε over P(X,Y) for 44850 pairs
--------------------------------------------------
max ε   : 5.007354
mean ε  : 2.770506
median ε: 2.894766
std ε   : 0.957452
min ε   : 0.449479


## Px - scaling 1

In [7]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------



import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from your filenames
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Px_scaling1/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# sanity check
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Determine number of classes U once, from all labels
# ----------------------------------------------------------------------
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# ----------------------------------------------------------------------
# Build joint Gaussian (mean, variance) summaries for P(X,Y)
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)        # (n, d_x)
    y = labels[k].astype(int)                          # (n,)
    # one-hot encode Y
    Y = np.eye(U, dtype=np.float64)[y]                 # (n, U)
    # joint samples Z = [X; Y]
    Z = np.concatenate([X, Y], axis=1)                 # (n, d_x + U)
    mu  = Z.mean(axis=0)                               # (d_x+U,)
    var = Z.var(axis=0)                                # (d_x+U,)
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """W2 between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    dm2 = np.sum((mu1 - mu2)**2)
    ds2 = np.sum((np.sqrt(var1) - np.sqrt(var2))**2)
    return np.sqrt(dm2 + ds2)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    # Euclidean on your saved descriptor profiles
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    # reference W2 on P(X,Y)
    mu1, var1 = gaussians[k1]
    mu2, var2 = gaussians[k2]
    ref_dist   = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print(f"\nWasserstein-2( P(X,Y) ) ε over {len(epsilons)} pairs")
print("-" * 40)
print(f"max ε   : {epsilons.max():.6f}")
print(f"mean ε  : {epsilons.mean():.6f}")
print(f"median ε: {np.median(epsilons):.6f}")
print(f"std ε   : {epsilons.std():.6f}")
print(f"min ε   : {epsilons.min():.6f}")

pairs: 100%|██████████| 44850/44850 [00:01<00:00, 25304.82it/s]


Wasserstein-2( P(X,Y) ) ε over 44850 pairs
----------------------------------------
max ε   : 26.101527
mean ε  : 16.571790
median ε: 23.930475
std ε   : 11.390280
min ε   : 0.000048


In [18]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Px_scaling1/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")


def flatten(x):
    # returns shape (n_samples, n_features)
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# Determine number of classes
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# Build joint histograms
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50
bins  = np.linspace(vmin, vmax, nbins+1)

histograms = {}
for k in descriptors:
    X_flat = flatten(features[k]).astype(np.float64)  # (n_samples, d)
    y      = labels[k].astype(int)                    # (n_samples,)
    joint_counts = np.zeros((U, nbins), dtype=np.float64)

    for u in range(U):
        mask = (y == u)           # shape (n_samples,)
        if not mask.any():
            continue
        # only flatten the samples of class u
        data_u = X_flat[mask].ravel()  
        counts, _ = np.histogram(data_u, bins=bins)
        joint_counts[u] = counts

    prob = joint_counts.ravel()
    total = prob.sum()
    if total > 0:
        prob /= total
    histograms[k] = prob

# Compute ε_js over all pairs
from scipy.spatial.distance import jensenshannon
import itertools
from tqdm import tqdm

pairs  = list(itertools.combinations(descriptors.keys(), 2))
eps_js = []

for k1, k2 in tqdm(pairs, desc="JSD P(X,Y) pairs"):
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])
    p  = histograms[k1]
    q  = histograms[k2]
    js = jensenshannon(p, q)
    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print(f"\nJensen–Shannon ε over P(X,Y) for {len(eps_js)} pairs")
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD P(X,Y) pairs: 100%|██████████| 44850/44850 [00:08<00:00, 5076.08it/s]


Jensen–Shannon ε over P(X,Y) for 44850 pairs
--------------------------------------------------
max ε   : 2.757768
mean ε  : 1.538091
median ε: 1.663716
std ε   : 0.536538
min ε   : 0.204589


## Px - scaling 2

In [8]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------



import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from your filenames
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Px_scaling2/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# sanity check
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Determine number of classes U once, from all labels
# ----------------------------------------------------------------------
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# ----------------------------------------------------------------------
# Build joint Gaussian (mean, variance) summaries for P(X,Y)
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)        # (n, d_x)
    y = labels[k].astype(int)                          # (n,)
    # one-hot encode Y
    Y = np.eye(U, dtype=np.float64)[y]                 # (n, U)
    # joint samples Z = [X; Y]
    Z = np.concatenate([X, Y], axis=1)                 # (n, d_x + U)
    mu  = Z.mean(axis=0)                               # (d_x+U,)
    var = Z.var(axis=0)                                # (d_x+U,)
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """W2 between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    dm2 = np.sum((mu1 - mu2)**2)
    ds2 = np.sum((np.sqrt(var1) - np.sqrt(var2))**2)
    return np.sqrt(dm2 + ds2)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    # Euclidean on your saved descriptor profiles
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    # reference W2 on P(X,Y)
    mu1, var1 = gaussians[k1]
    mu2, var2 = gaussians[k2]
    ref_dist   = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print(f"\nWasserstein-2( P(X,Y) ) ε over {len(epsilons)} pairs")
print("-" * 40)
print(f"max ε   : {epsilons.max():.6f}")
print(f"mean ε  : {epsilons.mean():.6f}")
print(f"median ε: {np.median(epsilons):.6f}")
print(f"std ε   : {epsilons.std():.6f}")
print(f"min ε   : {epsilons.min():.6f}")

pairs: 100%|██████████| 44850/44850 [00:01<00:00, 24890.11it/s]


Wasserstein-2( P(X,Y) ) ε over 44850 pairs
----------------------------------------
max ε   : 23.542973
mean ε  : 13.230794
median ε: 17.447226
std ε   : 9.294528
min ε   : 0.000042


In [19]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Px_scaling2/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")


def flatten(x):
    # returns shape (n_samples, n_features)
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# Determine number of classes
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# Build joint histograms
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50
bins  = np.linspace(vmin, vmax, nbins+1)

histograms = {}
for k in descriptors:
    X_flat = flatten(features[k]).astype(np.float64)  # (n_samples, d)
    y      = labels[k].astype(int)                    # (n_samples,)
    joint_counts = np.zeros((U, nbins), dtype=np.float64)

    for u in range(U):
        mask = (y == u)           # shape (n_samples,)
        if not mask.any():
            continue
        # only flatten the samples of class u
        data_u = X_flat[mask].ravel()  
        counts, _ = np.histogram(data_u, bins=bins)
        joint_counts[u] = counts

    prob = joint_counts.ravel()
    total = prob.sum()
    if total > 0:
        prob /= total
    histograms[k] = prob

# Compute ε_js over all pairs
from scipy.spatial.distance import jensenshannon
import itertools
from tqdm import tqdm

pairs  = list(itertools.combinations(descriptors.keys(), 2))
eps_js = []

for k1, k2 in tqdm(pairs, desc="JSD P(X,Y) pairs"):
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])
    p  = histograms[k1]
    q  = histograms[k2]
    js = jensenshannon(p, q)
    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print(f"\nJensen–Shannon ε over P(X,Y) for {len(eps_js)} pairs")
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD P(X,Y) pairs: 100%|██████████| 44850/44850 [00:10<00:00, 4479.99it/s]



Jensen–Shannon ε over P(X,Y) for 44850 pairs
--------------------------------------------------
max ε   : 10.485703
mean ε  : 4.746957
median ε: 3.420946
std ε   : 3.256622
min ε   : 0.096536


.

.

.

.

.

.

. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .

.

.

.

.

.

.



# Num. Epoch = 5


## Py - scaling 0

In [9]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------



import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from your filenames
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Py_scaling0/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# sanity check
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Determine number of classes U once, from all labels
# ----------------------------------------------------------------------
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# ----------------------------------------------------------------------
# Build joint Gaussian (mean, variance) summaries for P(X,Y)
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)        # (n, d_x)
    y = labels[k].astype(int)                          # (n,)
    # one-hot encode Y
    Y = np.eye(U, dtype=np.float64)[y]                 # (n, U)
    # joint samples Z = [X; Y]
    Z = np.concatenate([X, Y], axis=1)                 # (n, d_x + U)
    mu  = Z.mean(axis=0)                               # (d_x+U,)
    var = Z.var(axis=0)                                # (d_x+U,)
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """W2 between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    dm2 = np.sum((mu1 - mu2)**2)
    ds2 = np.sum((np.sqrt(var1) - np.sqrt(var2))**2)
    return np.sqrt(dm2 + ds2)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    # Euclidean on your saved descriptor profiles
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    # reference W2 on P(X,Y)
    mu1, var1 = gaussians[k1]
    mu2, var2 = gaussians[k2]
    ref_dist   = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print(f"\nWasserstein-2( P(X,Y) ) ε over {len(epsilons)} pairs")
print("-" * 40)
print(f"max ε   : {epsilons.max():.6f}")
print(f"mean ε  : {epsilons.mean():.6f}")
print(f"median ε: {np.median(epsilons):.6f}")
print(f"std ε   : {epsilons.std():.6f}")
print(f"min ε   : {epsilons.min():.6f}")

pairs: 100%|██████████| 44850/44850 [00:02<00:00, 22145.56it/s]


Wasserstein-2( P(X,Y) ) ε over 44850 pairs
----------------------------------------
max ε   : 14.230155
mean ε  : 8.028494
median ε: 7.827081
std ε   : 4.167748
min ε   : 0.740757


In [20]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Py_scaling0/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")


def flatten(x):
    # returns shape (n_samples, n_features)
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# Determine number of classes
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# Build joint histograms
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50
bins  = np.linspace(vmin, vmax, nbins+1)

histograms = {}
for k in descriptors:
    X_flat = flatten(features[k]).astype(np.float64)  # (n_samples, d)
    y      = labels[k].astype(int)                    # (n_samples,)
    joint_counts = np.zeros((U, nbins), dtype=np.float64)

    for u in range(U):
        mask = (y == u)           # shape (n_samples,)
        if not mask.any():
            continue
        # only flatten the samples of class u
        data_u = X_flat[mask].ravel()  
        counts, _ = np.histogram(data_u, bins=bins)
        joint_counts[u] = counts

    prob = joint_counts.ravel()
    total = prob.sum()
    if total > 0:
        prob /= total
    histograms[k] = prob

# Compute ε_js over all pairs
from scipy.spatial.distance import jensenshannon
import itertools
from tqdm import tqdm

pairs  = list(itertools.combinations(descriptors.keys(), 2))
eps_js = []

for k1, k2 in tqdm(pairs, desc="JSD P(X,Y) pairs"):
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])
    p  = histograms[k1]
    q  = histograms[k2]
    js = jensenshannon(p, q)
    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print(f"\nJensen–Shannon ε over P(X,Y) for {len(eps_js)} pairs")
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD P(X,Y) pairs: 100%|██████████| 44850/44850 [00:09<00:00, 4943.23it/s] 



Jensen–Shannon ε over P(X,Y) for 44850 pairs
--------------------------------------------------
max ε   : 15.999604
mean ε  : 9.937489
median ε: 11.122049
std ε   : 6.096687
min ε   : 0.002326


## Py - scaling 1


In [10]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------



import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from your filenames
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Py_scaling1/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# sanity check
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Determine number of classes U once, from all labels
# ----------------------------------------------------------------------
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# ----------------------------------------------------------------------
# Build joint Gaussian (mean, variance) summaries for P(X,Y)
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)        # (n, d_x)
    y = labels[k].astype(int)                          # (n,)
    # one-hot encode Y
    Y = np.eye(U, dtype=np.float64)[y]                 # (n, U)
    # joint samples Z = [X; Y]
    Z = np.concatenate([X, Y], axis=1)                 # (n, d_x + U)
    mu  = Z.mean(axis=0)                               # (d_x+U,)
    var = Z.var(axis=0)                                # (d_x+U,)
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """W2 between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    dm2 = np.sum((mu1 - mu2)**2)
    ds2 = np.sum((np.sqrt(var1) - np.sqrt(var2))**2)
    return np.sqrt(dm2 + ds2)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    # Euclidean on your saved descriptor profiles
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    # reference W2 on P(X,Y)
    mu1, var1 = gaussians[k1]
    mu2, var2 = gaussians[k2]
    ref_dist   = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print(f"\nWasserstein-2( P(X,Y) ) ε over {len(epsilons)} pairs")
print("-" * 40)
print(f"max ε   : {epsilons.max():.6f}")
print(f"mean ε  : {epsilons.mean():.6f}")
print(f"median ε: {np.median(epsilons):.6f}")
print(f"std ε   : {epsilons.std():.6f}")
print(f"min ε   : {epsilons.min():.6f}")

pairs: 100%|██████████| 44850/44850 [00:04<00:00, 10420.81it/s]


Wasserstein-2( P(X,Y) ) ε over 44850 pairs
----------------------------------------
max ε   : 7.857344
mean ε  : 4.029333
median ε: 3.713384
std ε   : 1.743229
min ε   : 0.003899


In [21]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Py_scaling1/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")


def flatten(x):
    # returns shape (n_samples, n_features)
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# Determine number of classes
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# Build joint histograms
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50
bins  = np.linspace(vmin, vmax, nbins+1)

histograms = {}
for k in descriptors:
    X_flat = flatten(features[k]).astype(np.float64)  # (n_samples, d)
    y      = labels[k].astype(int)                    # (n_samples,)
    joint_counts = np.zeros((U, nbins), dtype=np.float64)

    for u in range(U):
        mask = (y == u)           # shape (n_samples,)
        if not mask.any():
            continue
        # only flatten the samples of class u
        data_u = X_flat[mask].ravel()  
        counts, _ = np.histogram(data_u, bins=bins)
        joint_counts[u] = counts

    prob = joint_counts.ravel()
    total = prob.sum()
    if total > 0:
        prob /= total
    histograms[k] = prob

# Compute ε_js over all pairs
from scipy.spatial.distance import jensenshannon
import itertools
from tqdm import tqdm

pairs  = list(itertools.combinations(descriptors.keys(), 2))
eps_js = []

for k1, k2 in tqdm(pairs, desc="JSD P(X,Y) pairs"):
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])
    p  = histograms[k1]
    q  = histograms[k2]
    js = jensenshannon(p, q)
    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print(f"\nJensen–Shannon ε over P(X,Y) for {len(eps_js)} pairs")
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD P(X,Y) pairs: 100%|██████████| 44850/44850 [00:05<00:00, 8593.26it/s] 


Jensen–Shannon ε over P(X,Y) for 44850 pairs
--------------------------------------------------
max ε   : 9.546346
mean ε  : 6.407366
median ε: 6.705975
std ε   : 3.157018
min ε   : 0.000000


## Py - scaling 2

In [11]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------



import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from your filenames
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Py_scaling2/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# sanity check
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Determine number of classes U once, from all labels
# ----------------------------------------------------------------------
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# ----------------------------------------------------------------------
# Build joint Gaussian (mean, variance) summaries for P(X,Y)
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)        # (n, d_x)
    y = labels[k].astype(int)                          # (n,)
    # one-hot encode Y
    Y = np.eye(U, dtype=np.float64)[y]                 # (n, U)
    # joint samples Z = [X; Y]
    Z = np.concatenate([X, Y], axis=1)                 # (n, d_x + U)
    mu  = Z.mean(axis=0)                               # (d_x+U,)
    var = Z.var(axis=0)                                # (d_x+U,)
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """W2 between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    dm2 = np.sum((mu1 - mu2)**2)
    ds2 = np.sum((np.sqrt(var1) - np.sqrt(var2))**2)
    return np.sqrt(dm2 + ds2)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    # Euclidean on your saved descriptor profiles
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    # reference W2 on P(X,Y)
    mu1, var1 = gaussians[k1]
    mu2, var2 = gaussians[k2]
    ref_dist   = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print(f"\nWasserstein-2( P(X,Y) ) ε over {len(epsilons)} pairs")
print("-" * 40)
print(f"max ε   : {epsilons.max():.6f}")
print(f"mean ε  : {epsilons.mean():.6f}")
print(f"median ε: {np.median(epsilons):.6f}")
print(f"std ε   : {epsilons.std():.6f}")
print(f"min ε   : {epsilons.min():.6f}")

pairs: 100%|██████████| 44850/44850 [00:02<00:00, 16516.38it/s]


Wasserstein-2( P(X,Y) ) ε over 44850 pairs
----------------------------------------
max ε   : 10.497851
mean ε  : 6.757994
median ε: 7.309452
std ε   : 2.524178
min ε   : 0.720934


In [22]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Py_scaling2/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")


def flatten(x):
    # returns shape (n_samples, n_features)
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# Determine number of classes
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# Build joint histograms
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50
bins  = np.linspace(vmin, vmax, nbins+1)

histograms = {}
for k in descriptors:
    X_flat = flatten(features[k]).astype(np.float64)  # (n_samples, d)
    y      = labels[k].astype(int)                    # (n_samples,)
    joint_counts = np.zeros((U, nbins), dtype=np.float64)

    for u in range(U):
        mask = (y == u)           # shape (n_samples,)
        if not mask.any():
            continue
        # only flatten the samples of class u
        data_u = X_flat[mask].ravel()  
        counts, _ = np.histogram(data_u, bins=bins)
        joint_counts[u] = counts

    prob = joint_counts.ravel()
    total = prob.sum()
    if total > 0:
        prob /= total
    histograms[k] = prob

# Compute ε_js over all pairs
from scipy.spatial.distance import jensenshannon
import itertools
from tqdm import tqdm

pairs  = list(itertools.combinations(descriptors.keys(), 2))
eps_js = []

for k1, k2 in tqdm(pairs, desc="JSD P(X,Y) pairs"):
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])
    p  = histograms[k1]
    q  = histograms[k2]
    js = jensenshannon(p, q)
    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print(f"\nJensen–Shannon ε over P(X,Y) for {len(eps_js)} pairs")
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD P(X,Y) pairs: 100%|██████████| 44850/44850 [00:04<00:00, 9679.66it/s] 



Jensen–Shannon ε over P(X,Y) for 44850 pairs
--------------------------------------------------
max ε   : 12.293239
mean ε  : 9.330822
median ε: 12.095370
std ε   : 3.933741
min ε   : 0.000013


.

.

.

.

.

.

. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .

.

.

.

.

.

.

# Num. Epoch = 5

## P(X|Y) - scaling 0

In [23]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------



import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from your filenames
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Pxy_scaling0/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# sanity check
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Determine number of classes U once, from all labels
# ----------------------------------------------------------------------
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# ----------------------------------------------------------------------
# Build joint Gaussian (mean, variance) summaries for P(X,Y)
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)        # (n, d_x)
    y = labels[k].astype(int)                          # (n,)
    # one-hot encode Y
    Y = np.eye(U, dtype=np.float64)[y]                 # (n, U)
    # joint samples Z = [X; Y]
    Z = np.concatenate([X, Y], axis=1)                 # (n, d_x + U)
    mu  = Z.mean(axis=0)                               # (d_x+U,)
    var = Z.var(axis=0)                                # (d_x+U,)
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """W2 between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    dm2 = np.sum((mu1 - mu2)**2)
    ds2 = np.sum((np.sqrt(var1) - np.sqrt(var2))**2)
    return np.sqrt(dm2 + ds2)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    # Euclidean on your saved descriptor profiles
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    # reference W2 on P(X,Y)
    mu1, var1 = gaussians[k1]
    mu2, var2 = gaussians[k2]
    ref_dist   = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print(f"\nWasserstein-2( P(X,Y) ) ε over {len(epsilons)} pairs")
print("-" * 40)
print(f"max ε   : {epsilons.max():.6f}")
print(f"mean ε  : {epsilons.mean():.6f}")
print(f"median ε: {np.median(epsilons):.6f}")
print(f"std ε   : {epsilons.std():.6f}")
print(f"min ε   : {epsilons.min():.6f}")

pairs: 100%|██████████| 44850/44850 [00:04<00:00, 9230.51it/s] 


Wasserstein-2( P(X,Y) ) ε over 44850 pairs
----------------------------------------
max ε   : 9.128509
mean ε  : 4.137777
median ε: 4.235593
std ε   : 2.726843
min ε   : 0.000042


In [24]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Pxy_scaling0/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")


def flatten(x):
    # returns shape (n_samples, n_features)
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# Determine number of classes
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# Build joint histograms
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50
bins  = np.linspace(vmin, vmax, nbins+1)

histograms = {}
for k in descriptors:
    X_flat = flatten(features[k]).astype(np.float64)  # (n_samples, d)
    y      = labels[k].astype(int)                    # (n_samples,)
    joint_counts = np.zeros((U, nbins), dtype=np.float64)

    for u in range(U):
        mask = (y == u)           # shape (n_samples,)
        if not mask.any():
            continue
        # only flatten the samples of class u
        data_u = X_flat[mask].ravel()  
        counts, _ = np.histogram(data_u, bins=bins)
        joint_counts[u] = counts

    prob = joint_counts.ravel()
    total = prob.sum()
    if total > 0:
        prob /= total
    histograms[k] = prob

# Compute ε_js over all pairs
from scipy.spatial.distance import jensenshannon
import itertools
from tqdm import tqdm

pairs  = list(itertools.combinations(descriptors.keys(), 2))
eps_js = []

for k1, k2 in tqdm(pairs, desc="JSD P(X,Y) pairs"):
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])
    p  = histograms[k1]
    q  = histograms[k2]
    js = jensenshannon(p, q)
    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print(f"\nJensen–Shannon ε over P(X,Y) for {len(eps_js)} pairs")
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD P(X,Y) pairs: 100%|██████████| 44850/44850 [00:02<00:00, 18953.85it/s]


Jensen–Shannon ε over P(X,Y) for 44850 pairs
--------------------------------------------------
max ε   : 24.113203
mean ε  : 14.150132
median ε: 22.283677
std ε   : 9.563232
min ε   : 0.150943


## P(X|Y) - scaling 1

In [13]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------



import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from your filenames
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Pxy_scaling1/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# sanity check
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Determine number of classes U once, from all labels
# ----------------------------------------------------------------------
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# ----------------------------------------------------------------------
# Build joint Gaussian (mean, variance) summaries for P(X,Y)
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)        # (n, d_x)
    y = labels[k].astype(int)                          # (n,)
    # one-hot encode Y
    Y = np.eye(U, dtype=np.float64)[y]                 # (n, U)
    # joint samples Z = [X; Y]
    Z = np.concatenate([X, Y], axis=1)                 # (n, d_x + U)
    mu  = Z.mean(axis=0)                               # (d_x+U,)
    var = Z.var(axis=0)                                # (d_x+U,)
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """W2 between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    dm2 = np.sum((mu1 - mu2)**2)
    ds2 = np.sum((np.sqrt(var1) - np.sqrt(var2))**2)
    return np.sqrt(dm2 + ds2)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    # Euclidean on your saved descriptor profiles
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    # reference W2 on P(X,Y)
    mu1, var1 = gaussians[k1]
    mu2, var2 = gaussians[k2]
    ref_dist   = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print(f"\nWasserstein-2( P(X,Y) ) ε over {len(epsilons)} pairs")
print("-" * 40)
print(f"max ε   : {epsilons.max():.6f}")
print(f"mean ε  : {epsilons.mean():.6f}")
print(f"median ε: {np.median(epsilons):.6f}")
print(f"std ε   : {epsilons.std():.6f}")
print(f"min ε   : {epsilons.min():.6f}")

pairs: 100%|██████████| 44850/44850 [00:04<00:00, 11012.19it/s]


Wasserstein-2( P(X,Y) ) ε over 44850 pairs
----------------------------------------
max ε   : 17.151852
mean ε  : 4.372418
median ε: 0.694197
std ε   : 5.561130
min ε   : 0.000005


In [25]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Pxy_scaling1/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")


def flatten(x):
    # returns shape (n_samples, n_features)
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# Determine number of classes
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# Build joint histograms
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50
bins  = np.linspace(vmin, vmax, nbins+1)

histograms = {}
for k in descriptors:
    X_flat = flatten(features[k]).astype(np.float64)  # (n_samples, d)
    y      = labels[k].astype(int)                    # (n_samples,)
    joint_counts = np.zeros((U, nbins), dtype=np.float64)

    for u in range(U):
        mask = (y == u)           # shape (n_samples,)
        if not mask.any():
            continue
        # only flatten the samples of class u
        data_u = X_flat[mask].ravel()  
        counts, _ = np.histogram(data_u, bins=bins)
        joint_counts[u] = counts

    prob = joint_counts.ravel()
    total = prob.sum()
    if total > 0:
        prob /= total
    histograms[k] = prob

# Compute ε_js over all pairs
from scipy.spatial.distance import jensenshannon
import itertools
from tqdm import tqdm

pairs  = list(itertools.combinations(descriptors.keys(), 2))
eps_js = []

for k1, k2 in tqdm(pairs, desc="JSD P(X,Y) pairs"):
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])
    p  = histograms[k1]
    q  = histograms[k2]
    js = jensenshannon(p, q)
    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print(f"\nJensen–Shannon ε over P(X,Y) for {len(eps_js)} pairs")
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD P(X,Y) pairs: 100%|██████████| 44850/44850 [00:02<00:00, 16629.32it/s]


Jensen–Shannon ε over P(X,Y) for 44850 pairs
--------------------------------------------------
max ε   : 20.464851
mean ε  : 12.848718
median ε: 13.690525
std ε   : 7.092164
min ε   : 0.043027


## P(X|Y) - scaling 2

In [14]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------



import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from your filenames
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Pxy_scaling2/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# sanity check
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Determine number of classes U once, from all labels
# ----------------------------------------------------------------------
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# ----------------------------------------------------------------------
# Build joint Gaussian (mean, variance) summaries for P(X,Y)
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)        # (n, d_x)
    y = labels[k].astype(int)                          # (n,)
    # one-hot encode Y
    Y = np.eye(U, dtype=np.float64)[y]                 # (n, U)
    # joint samples Z = [X; Y]
    Z = np.concatenate([X, Y], axis=1)                 # (n, d_x + U)
    mu  = Z.mean(axis=0)                               # (d_x+U,)
    var = Z.var(axis=0)                                # (d_x+U,)
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """W2 between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    dm2 = np.sum((mu1 - mu2)**2)
    ds2 = np.sum((np.sqrt(var1) - np.sqrt(var2))**2)
    return np.sqrt(dm2 + ds2)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    # Euclidean on your saved descriptor profiles
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    # reference W2 on P(X,Y)
    mu1, var1 = gaussians[k1]
    mu2, var2 = gaussians[k2]
    ref_dist   = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print(f"\nWasserstein-2( P(X,Y) ) ε over {len(epsilons)} pairs")
print("-" * 40)
print(f"max ε   : {epsilons.max():.6f}")
print(f"mean ε  : {epsilons.mean():.6f}")
print(f"median ε: {np.median(epsilons):.6f}")
print(f"std ε   : {epsilons.std():.6f}")
print(f"min ε   : {epsilons.min():.6f}")

pairs: 100%|██████████| 44850/44850 [00:05<00:00, 8114.41it/s] 


Wasserstein-2( P(X,Y) ) ε over 44850 pairs
----------------------------------------
max ε   : 10.992137
mean ε  : 4.936504
median ε: 4.860213
std ε   : 2.559522
min ε   : 0.000077


In [26]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Pxy_scaling2/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")


def flatten(x):
    # returns shape (n_samples, n_features)
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# Determine number of classes
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# Build joint histograms
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50
bins  = np.linspace(vmin, vmax, nbins+1)

histograms = {}
for k in descriptors:
    X_flat = flatten(features[k]).astype(np.float64)  # (n_samples, d)
    y      = labels[k].astype(int)                    # (n_samples,)
    joint_counts = np.zeros((U, nbins), dtype=np.float64)

    for u in range(U):
        mask = (y == u)           # shape (n_samples,)
        if not mask.any():
            continue
        # only flatten the samples of class u
        data_u = X_flat[mask].ravel()  
        counts, _ = np.histogram(data_u, bins=bins)
        joint_counts[u] = counts

    prob = joint_counts.ravel()
    total = prob.sum()
    if total > 0:
        prob /= total
    histograms[k] = prob

# Compute ε_js over all pairs
from scipy.spatial.distance import jensenshannon
import itertools
from tqdm import tqdm

pairs  = list(itertools.combinations(descriptors.keys(), 2))
eps_js = []

for k1, k2 in tqdm(pairs, desc="JSD P(X,Y) pairs"):
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])
    p  = histograms[k1]
    q  = histograms[k2]
    js = jensenshannon(p, q)
    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print(f"\nJensen–Shannon ε over P(X,Y) for {len(eps_js)} pairs")
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD P(X,Y) pairs: 100%|██████████| 44850/44850 [00:02<00:00, 15627.16it/s]


Jensen–Shannon ε over P(X,Y) for 44850 pairs
--------------------------------------------------
max ε   : 25.375866
mean ε  : 14.485295
median ε: 12.707214
std ε   : 8.285014
min ε   : 0.155591


.

.

.

.

.

.

. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .

.

.

.

.

.

.

# Num. Epoch = 5

## P(Y|X) - scaling 0

In [15]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------



import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from your filenames
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Pyx_scaling0/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# sanity check
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Determine number of classes U once, from all labels
# ----------------------------------------------------------------------
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# ----------------------------------------------------------------------
# Build joint Gaussian (mean, variance) summaries for P(X,Y)
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)        # (n, d_x)
    y = labels[k].astype(int)                          # (n,)
    # one-hot encode Y
    Y = np.eye(U, dtype=np.float64)[y]                 # (n, U)
    # joint samples Z = [X; Y]
    Z = np.concatenate([X, Y], axis=1)                 # (n, d_x + U)
    mu  = Z.mean(axis=0)                               # (d_x+U,)
    var = Z.var(axis=0)                                # (d_x+U,)
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """W2 between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    dm2 = np.sum((mu1 - mu2)**2)
    ds2 = np.sum((np.sqrt(var1) - np.sqrt(var2))**2)
    return np.sqrt(dm2 + ds2)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    # Euclidean on your saved descriptor profiles
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    # reference W2 on P(X,Y)
    mu1, var1 = gaussians[k1]
    mu2, var2 = gaussians[k2]
    ref_dist   = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print(f"\nWasserstein-2( P(X,Y) ) ε over {len(epsilons)} pairs")
print("-" * 40)
print(f"max ε   : {epsilons.max():.6f}")
print(f"mean ε  : {epsilons.mean():.6f}")
print(f"median ε: {np.median(epsilons):.6f}")
print(f"std ε   : {epsilons.std():.6f}")
print(f"min ε   : {epsilons.min():.6f}")

pairs: 100%|██████████| 44850/44850 [00:02<00:00, 16133.91it/s]


Wasserstein-2( P(X,Y) ) ε over 44850 pairs
----------------------------------------
max ε   : 2.815795
mean ε  : 1.327561
median ε: 1.443142
std ε   : 0.521781
min ε   : 0.000172


In [27]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Pyx_scaling0/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")


def flatten(x):
    # returns shape (n_samples, n_features)
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# Determine number of classes
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# Build joint histograms
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50
bins  = np.linspace(vmin, vmax, nbins+1)

histograms = {}
for k in descriptors:
    X_flat = flatten(features[k]).astype(np.float64)  # (n_samples, d)
    y      = labels[k].astype(int)                    # (n_samples,)
    joint_counts = np.zeros((U, nbins), dtype=np.float64)

    for u in range(U):
        mask = (y == u)           # shape (n_samples,)
        if not mask.any():
            continue
        # only flatten the samples of class u
        data_u = X_flat[mask].ravel()  
        counts, _ = np.histogram(data_u, bins=bins)
        joint_counts[u] = counts

    prob = joint_counts.ravel()
    total = prob.sum()
    if total > 0:
        prob /= total
    histograms[k] = prob

# Compute ε_js over all pairs
from scipy.spatial.distance import jensenshannon
import itertools
from tqdm import tqdm

pairs  = list(itertools.combinations(descriptors.keys(), 2))
eps_js = []

for k1, k2 in tqdm(pairs, desc="JSD P(X,Y) pairs"):
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])
    p  = histograms[k1]
    q  = histograms[k2]
    js = jensenshannon(p, q)
    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print(f"\nJensen–Shannon ε over P(X,Y) for {len(eps_js)} pairs")
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD P(X,Y) pairs: 100%|██████████| 44850/44850 [00:02<00:00, 17811.12it/s]


Jensen–Shannon ε over P(X,Y) for 44850 pairs
--------------------------------------------------
max ε   : 3.390307
mean ε  : 2.161086
median ε: 2.314057
std ε   : 0.510472
min ε   : 0.434444


## P(Y|X) - scaling 1

In [16]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------



import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from your filenames
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Pyx_scaling1/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# sanity check
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Determine number of classes U once, from all labels
# ----------------------------------------------------------------------
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# ----------------------------------------------------------------------
# Build joint Gaussian (mean, variance) summaries for P(X,Y)
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)        # (n, d_x)
    y = labels[k].astype(int)                          # (n,)
    # one-hot encode Y
    Y = np.eye(U, dtype=np.float64)[y]                 # (n, U)
    # joint samples Z = [X; Y]
    Z = np.concatenate([X, Y], axis=1)                 # (n, d_x + U)
    mu  = Z.mean(axis=0)                               # (d_x+U,)
    var = Z.var(axis=0)                                # (d_x+U,)
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """W2 between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    dm2 = np.sum((mu1 - mu2)**2)
    ds2 = np.sum((np.sqrt(var1) - np.sqrt(var2))**2)
    return np.sqrt(dm2 + ds2)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    # Euclidean on your saved descriptor profiles
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    # reference W2 on P(X,Y)
    mu1, var1 = gaussians[k1]
    mu2, var2 = gaussians[k2]
    ref_dist   = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print(f"\nWasserstein-2( P(X,Y) ) ε over {len(epsilons)} pairs")
print("-" * 40)
print(f"max ε   : {epsilons.max():.6f}")
print(f"mean ε  : {epsilons.mean():.6f}")
print(f"median ε: {np.median(epsilons):.6f}")
print(f"std ε   : {epsilons.std():.6f}")
print(f"min ε   : {epsilons.min():.6f}")

pairs: 100%|██████████| 44850/44850 [00:06<00:00, 6716.91it/s] 


Wasserstein-2( P(X,Y) ) ε over 44850 pairs
----------------------------------------
max ε   : 3.464037
mean ε  : 1.743970
median ε: 1.778193
std ε   : 0.551097
min ε   : 0.001346


In [28]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Pyx_scaling1/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")


def flatten(x):
    # returns shape (n_samples, n_features)
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# Determine number of classes
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# Build joint histograms
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50
bins  = np.linspace(vmin, vmax, nbins+1)

histograms = {}
for k in descriptors:
    X_flat = flatten(features[k]).astype(np.float64)  # (n_samples, d)
    y      = labels[k].astype(int)                    # (n_samples,)
    joint_counts = np.zeros((U, nbins), dtype=np.float64)

    for u in range(U):
        mask = (y == u)           # shape (n_samples,)
        if not mask.any():
            continue
        # only flatten the samples of class u
        data_u = X_flat[mask].ravel()  
        counts, _ = np.histogram(data_u, bins=bins)
        joint_counts[u] = counts

    prob = joint_counts.ravel()
    total = prob.sum()
    if total > 0:
        prob /= total
    histograms[k] = prob

# Compute ε_js over all pairs
from scipy.spatial.distance import jensenshannon
import itertools
from tqdm import tqdm

pairs  = list(itertools.combinations(descriptors.keys(), 2))
eps_js = []

for k1, k2 in tqdm(pairs, desc="JSD P(X,Y) pairs"):
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])
    p  = histograms[k1]
    q  = histograms[k2]
    js = jensenshannon(p, q)
    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print(f"\nJensen–Shannon ε over P(X,Y) for {len(eps_js)} pairs")
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD P(X,Y) pairs: 100%|██████████| 44850/44850 [00:02<00:00, 16000.11it/s]


Jensen–Shannon ε over P(X,Y) for 44850 pairs
--------------------------------------------------
max ε   : 4.276997
mean ε  : 2.570027
median ε: 2.611979
std ε   : 0.536816
min ε   : 0.444015


## P(Y|X) - scaling 2

In [17]:
# ----------------------------------------------------------------------
# Wasserstein-2 distance
# ----------------------------------------------------------------------



import os, re, glob, itertools
import numpy as np
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from your filenames
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Pyx_scaling2/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

# sanity check
missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")

# ----------------------------------------------------------------------
# Helper: flatten to [n_samples, n_features]
# ----------------------------------------------------------------------
def flatten(x):
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# ----------------------------------------------------------------------
# Determine number of classes U once, from all labels
# ----------------------------------------------------------------------
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# ----------------------------------------------------------------------
# Build joint Gaussian (mean, variance) summaries for P(X,Y)
# ----------------------------------------------------------------------
gaussians = {}
for k in descriptors.keys():
    X = flatten(features[k]).astype(np.float64)        # (n, d_x)
    y = labels[k].astype(int)                          # (n,)
    # one-hot encode Y
    Y = np.eye(U, dtype=np.float64)[y]                 # (n, U)
    # joint samples Z = [X; Y]
    Z = np.concatenate([X, Y], axis=1)                 # (n, d_x + U)
    mu  = Z.mean(axis=0)                               # (d_x+U,)
    var = Z.var(axis=0)                                # (d_x+U,)
    gaussians[k] = (mu, var)

# ----------------------------------------------------------------------
# Wasserstein-2 between diagonal Gaussians
# ----------------------------------------------------------------------
def w2_diag(mu1, var1, mu2, var2):
    """W2 between N(mu1, diag(var1)) and N(mu2, diag(var2))."""
    dm2 = np.sum((mu1 - mu2)**2)
    ds2 = np.sum((np.sqrt(var1) - np.sqrt(var2))**2)
    return np.sqrt(dm2 + ds2)

# ----------------------------------------------------------------------
# Compute ε for every unordered pair of (client, round) profiles
# ----------------------------------------------------------------------
pairs    = list(itertools.combinations(descriptors.keys(), 2))
epsilons = []

for k1, k2 in tqdm(pairs, desc="pairs"):
    # Euclidean on your saved descriptor profiles
    profile_dist = np.linalg.norm(descriptors[k1] - descriptors[k2])
    # reference W2 on P(X,Y)
    mu1, var1 = gaussians[k1]
    mu2, var2 = gaussians[k2]
    ref_dist   = w2_diag(mu1, var1, mu2, var2)
    epsilons.append(abs(profile_dist - ref_dist))

epsilons = np.array(epsilons)
print(f"\nWasserstein-2( P(X,Y) ) ε over {len(epsilons)} pairs")
print("-" * 40)
print(f"max ε   : {epsilons.max():.6f}")
print(f"mean ε  : {epsilons.mean():.6f}")
print(f"median ε: {np.median(epsilons):.6f}")
print(f"std ε   : {epsilons.std():.6f}")
print(f"min ε   : {epsilons.min():.6f}")

pairs: 100%|██████████| 44850/44850 [00:05<00:00, 7859.81it/s] 


Wasserstein-2( P(X,Y) ) ε over 44850 pairs
----------------------------------------
max ε   : 3.828891
mean ε  : 2.033825
median ε: 2.054456
std ε   : 0.589545
min ε   : 0.004410


In [29]:
# ----------------------------------------------------------------------
# jensenshannon distance
# ----------------------------------------------------------------------


import os, re, glob, itertools
import numpy as np
from scipy.spatial.distance import jensenshannon
from tqdm import tqdm

# ----------------------------------------------------------------------
# Regex to capture client-id and round from the file names you produced
# ----------------------------------------------------------------------
pattern = re.compile(r"client_(\d+)_round_(\d+)")

descriptors, features, labels = {}, {}, {}

for f in glob.glob("cifar10_epsilon/temp_Pyx_scaling2/*.npy"):
    m = pattern.search(f)
    if not m:
        continue
    key = tuple(map(int, m.groups()))
    if "descriptor" in f:
        descriptors[key] = np.load(f)
    elif "features" in f:
        features[key] = np.load(f)
    elif "labels" in f:
        labels[key] = np.load(f)

missing = [k for k in descriptors if k not in features or k not in labels]
if missing:
    raise RuntimeError(f"Missing .npy files for {missing}")


def flatten(x):
    # returns shape (n_samples, n_features)
    return x.reshape(x.shape[0], -1) if x.ndim > 2 else x

# Determine number of classes
all_labels = np.concatenate(list(labels.values()))
U = int(all_labels.max()) + 1

# Build joint histograms
all_vals = np.concatenate([flatten(features[k]).ravel() for k in features])
vmin, vmax = all_vals.min(), all_vals.max()
nbins = 50
bins  = np.linspace(vmin, vmax, nbins+1)

histograms = {}
for k in descriptors:
    X_flat = flatten(features[k]).astype(np.float64)  # (n_samples, d)
    y      = labels[k].astype(int)                    # (n_samples,)
    joint_counts = np.zeros((U, nbins), dtype=np.float64)

    for u in range(U):
        mask = (y == u)           # shape (n_samples,)
        if not mask.any():
            continue
        # only flatten the samples of class u
        data_u = X_flat[mask].ravel()  
        counts, _ = np.histogram(data_u, bins=bins)
        joint_counts[u] = counts

    prob = joint_counts.ravel()
    total = prob.sum()
    if total > 0:
        prob /= total
    histograms[k] = prob

# Compute ε_js over all pairs
from scipy.spatial.distance import jensenshannon
import itertools
from tqdm import tqdm

pairs  = list(itertools.combinations(descriptors.keys(), 2))
eps_js = []

for k1, k2 in tqdm(pairs, desc="JSD P(X,Y) pairs"):
    dp = np.linalg.norm(descriptors[k1] - descriptors[k2])
    p  = histograms[k1]
    q  = histograms[k2]
    js = jensenshannon(p, q)
    eps_js.append(abs(dp - js))

eps_js = np.array(eps_js)
print(f"\nJensen–Shannon ε over P(X,Y) for {len(eps_js)} pairs")
print("-" * 50)
print(f"max ε   : {eps_js.max():.6f}")
print(f"mean ε  : {eps_js.mean():.6f}")
print(f"median ε: {np.median(eps_js):.6f}")
print(f"std ε   : {eps_js.std():.6f}")
print(f"min ε   : {eps_js.min():.6f}")

JSD P(X,Y) pairs: 100%|██████████| 44850/44850 [00:14<00:00, 3067.21it/s]


Jensen–Shannon ε over P(X,Y) for 44850 pairs
--------------------------------------------------
max ε   : 4.483705
mean ε  : 2.857944
median ε: 2.873607
std ε   : 0.579394
min ε   : 0.445179
